In [1]:
:set -XRankNTypes
:set -XScopedTypeVariables
putStrLn "Setup done."

Setup done.

# 🔧 Трансформеры монад в Haskell
## Определение, категорные основания, практические инструменты

> *"Монады как скафандры: каждый делает одно хорошо. Трансформеры — это скафандры, которые можно надевать один на другой."*

## Содержание

1. [Мотивация: проблема комбинирования монад](#мотивация)
2. [Категорное определение трансформера монады](#категорное-определение)
3. [Класс типов MonadTrans](#monadtrans)
4. [MaybeT — обработка отказов](#maybet)
5. [ExceptT — обработка ошибок с информацией](#exceptt)
6. [StateT — состояние в стеке](#statet)
7. [ReaderT — окружение в стеке](#readert)
8. [WriterT — журналирование в стеке](#writert)
9. [ContT — продолжения](#contt)
10. [Интересные комбинации: стеки трансформеров](#stacks)
11. [mtl-стиль: классы для стеков](#mtl)
12. [Реальные примеры: парсер, интерпретатор, web-handler](#реальные-примеры)

## 1️⃣ Мотивация: проблема комбинирования монад <a id="мотивация"></a>

Монады прекрасно инкапсулируют один эффект. Но в реальных программах эффекты **накапливаются**:
- Читаем конфигурацию (**Reader**)
- Ведём лог (**Writer**)
- Храним состояние (**State**)
- Обрабатываем ошибки (**Either/Maybe**)
- Всё это в IO

Наивное вложение монад не работает: `Maybe (State s (Either e a))` — монада только если мы вручную протаскиваем все слои.

**Трансформер монады** — это функтор `t :: (* -> *) -> (* -> *)`, который берёт монаду `m` и создаёт новую монаду `t m`, добавляя один эффект.

In [2]:
-- Проблема: нельзя просто вложить монады
-- Если у нас есть Maybe и IO — нам нужен MonadTrans

-- Наивная попытка: работает для конкретных монад, но некомпозируемо
safeDiv :: Int -> Int -> Maybe Int
safeDiv _ 0 = Nothing
safeDiv x y = Just (x `div` y)

-- Хотим safeDiv в IO — придётся вручную тянуть Maybe через IO
safeDivIO :: Int -> Int -> IO (Maybe Int)
safeDivIO x y = return (safeDiv x y)

-- Это не монада! bind нужно писать вручную:
-- Нет автоматического do-синтаксиса для IO (Maybe a)

putStrLn "Проблема вложенных монад показана"

Проблема вложенных монад показана

### Решение: трансформеры

Трансформер `MaybeT m a` — это обёртка над `m (Maybe a)`, которая уже является монадой:

```
newtype MaybeT m a = MaybeT { runMaybeT :: m (Maybe a) }
```

Теперь мы можем писать `do`-нотацию и автоматически получать оба эффекта.

## 2️⃣ Категорное определение трансформера монады <a id="категорное-определение"></a>

### Монада как эндофунктор с двумя естественными преобразованиями

В категории **Hask** монада — это тройка `(T, η, μ)`:
- `T : Hask → Hask` — эндофунктор
- `η : Id ⟹ T` — единица (unit/return)
- `μ : T² ⟹ T` — умножение (join)

удовлетворяющая законам ассоциативности и единицы.

### Трансформер монады

**Трансформер монады** `t` — это отображение монад в монады:

`t : Monad m ⟹ Monad (t m)`

с функцией подъёма (lift):

`lift : m a → t m a`

которая должна сохранять структуру монады (быть морфизмом монад).

### Категория монад

Рассмотрим **категорию монад** `Mon(Hask)`:
- Объекты: монады над Hask
- Морфизмы: морфизмы монад (естественные преобразования, сохраняющие η и μ)

Трансформер монады `t` — это **эндофунктор** на `Mon(Hask)`!

In [3]:
-- Класс MonadTrans фиксирует это математически
import Control.Monad.Trans.Class

-- class MonadTrans t where
--   lift :: Monad m => m a -> t m a
-- Законы:
-- 1. lift . return  = return          (сохраняет единицу)
-- 2. lift (m >>= f) = lift m >>= (lift . f)   (сохраняет bind)

-- Проверим законы на примере MaybeT
import Control.Monad.Trans.Maybe

-- Закон 1: lift . return = return
-- lift (return x) :: MaybeT IO x
-- должно быть равно return x :: MaybeT IO x

example1 :: IO ()
example1 = do
  r1 <- runMaybeT (lift (return 42) :: MaybeT IO Int)
  r2 <- runMaybeT (return 42 :: MaybeT IO Int)
  putStrLn $ "lift (return 42) = " ++ show r1
  putStrLn $ "return 42        = " ++ show r2
  putStrLn $ "Законы совпадают: " ++ show (r1 == r2)

example1

lift (return 42) = Just 42
return 42        = Just 42
Законы совпадают: True

### Диаграмма: структура трансформера





Трансформер монады `t` создаёт **слоистую** структуру: внешний слой добавляет эффект, внутренняя монада `m` передаётся как параметр.

In [ ]:
-- Diagram: monad transformer structure
mtDiagramSvg :: String
mtDiagramSvg = unlines
  [ "<svg xmlns=\"http://www.w3.org/2000/svg\" width=\"480\" height=\"280\" viewBox=\"0 0 480 280\" font-family=\"monospace,Arial,sans-serif\">"
  , "  <rect width=\"480\" height=\"280\" fill=\"#ffffff\" rx=\"8\"/>"
  , "  <rect x=\"30\" y=\"30\" width=\"420\" height=\"220\" fill=\"#e0f2fe\" stroke=\"#2563eb\" stroke-width=\"2\" rx=\"10\"/>"
  , "  <text x=\"240\" y=\"55\" text-anchor=\"middle\" font-size=\"13\" font-weight=\"bold\" fill=\"#1e40af\">t m a  (transformer)</text>"
  , "  <rect x=\"80\" y=\"80\" width=\"320\" height=\"140\" fill=\"#dcfce7\" stroke=\"#34d399\" stroke-width=\"2\" rx=\"8\"/>"
  , "  <text x=\"240\" y=\"105\" text-anchor=\"middle\" font-size=\"12\" font-weight=\"bold\" fill=\"#065f46\">m a  (base monad)</text>"
  , "  <rect x=\"170\" y=\"125\" width=\"140\" height=\"60\" fill=\"#fef3c7\" stroke=\"#fb923c\" stroke-width=\"2\" rx=\"6\"/>"
  , "  <text x=\"240\" y=\"150\" text-anchor=\"middle\" font-size=\"14\" font-weight=\"bold\" fill=\"#92400e\">a</text>"
  , "  <text x=\"240\" y=\"170\" text-anchor=\"middle\" font-size=\"10\" fill=\"#64748b\">(value)</text>"
  , "  <text x=\"22\" y=\"145\" font-size=\"10\" font-weight=\"bold\" fill=\"#64748b\">lift</text>"
  , "  <text x=\"240\" y=\"265\" text-anchor=\"middle\" font-size=\"10\" fill=\"#64748b\">lift :: m a -&gt; t m a   (monad morphism)</text>"
  , "</svg>"
  ]

writeFile "/tmp/mt_diagram.svg" mtDiagramSvg
putStrLn "Diagram saved."

In [ ]:
:! cp /tmp/mt_diagram.svg /home/jovyan/src/mt_diagram.svg && echo "OK"

## 3️⃣ Класс типов MonadTrans <a id="monadtrans"></a>

```haskell
class MonadTrans t where
  lift :: Monad m => m a -> t m a
```

Законы `lift` (морфизм монад):
1. **Единица:** `lift (return a) = return a`
2. **Bind:** `lift (m >>= f) = lift m >>= (lift . f)`

### Важные классы mtl

| Класс | Что добавляет |
|-------|--------------|
| `MonadTrans t` | базовый `lift` |
| `MonadIO m` | `liftIO :: IO a -> m a` |
| `MonadReader r m` | `ask`, `local` |
| `MonadState s m` | `get`, `put`, `modify` |
| `MonadWriter w m` | `tell`, `listen` |
| `MonadError e m` | `throwError`, `catchError` |

In [ ]:
{-# LANGUAGE FlexibleContexts #-}
import Control.Monad.Trans.Class (lift)
import Control.Monad.IO.Class (liftIO)
import Control.Monad.Trans.Maybe (MaybeT(..), runMaybeT)
import Control.Monad.Trans.Except (ExceptT(..), runExceptT, throwE, catchE)
import Control.Monad.Trans.State (StateT(..), runStateT, get, put, modify)
import Control.Monad.Trans.Reader (ReaderT(..), runReaderT, ask, local)
import Control.Monad.Trans.Writer (WriterT(..), runWriterT, tell)

putStrLn "Все трансформеры импортированы!"

## 4️⃣ MaybeT — обработка отказов <a id="maybet"></a>

```haskell
newtype MaybeT m a = MaybeT { runMaybeT :: m (Maybe a) }
```

**MaybeT** добавляет возможность «провала» (`Nothing`) поверх любой монады `m`.

### Когда использовать
- Поиск/выборка которая может не найти результат
- Необязательные шаги в вычислении
- Ранний выход без ошибки

In [ ]:
-- Пример 1: Безопасный поиск в IO
import qualified Data.Map.Strict as Map
import Data.Map.Strict (Map)

type UsersDB = Map String Int  -- имя -> возраст

usersDB :: UsersDB
usersDB = Map.fromList [("Alice", 30), ("Bob", 25), ("Charlie", 35)]

-- Ищем пользователя и проверяем возраст
lookupUser :: String -> MaybeT IO Int
lookupUser name = do
  age <- MaybeT (return (Map.lookup name usersDB))
  liftIO $ putStrLn $ "Нашли: " ++ name ++ ", возраст " ++ show age
  return age

checkAdult :: String -> MaybeT IO String
checkAdult name = do
  age <- lookupUser name
  if age >= 18
    then return $ name ++ " — совершеннолетний"
    else MaybeT (return Nothing)  -- провал

main1 :: IO ()
main1 = do
  r1 <- runMaybeT (checkAdult "Alice")
  putStrLn $ "Alice: " ++ show r1
  r2 <- runMaybeT (checkAdult "Zoro")  -- не найден
  putStrLn $ "Zoro: " ++ show r2

main1

In [ ]:
-- Пример 2: Цепочка безопасных операций
import Data.Maybe (fromMaybe)

-- Безопасные операции
safeHead :: [a] -> Maybe a
safeHead [] = Nothing
safeHead (x:_) = Just x

safeLast :: [a] -> Maybe a
safeLast [] = Nothing
safeLast xs = Just (last xs)

safeIndex :: [a] -> Int -> Maybe a
safeIndex xs i
  | i < 0 || i >= length xs = Nothing
  | otherwise = Just (xs !! i)

-- MaybeT над списком! Нетривиальная монада
-- MaybeT [] — это список Maybe, т.е. вычисление с недетерминизмом И отказами
findBothEnds :: [Int] -> MaybeT [] (Int, Int)
findBothEnds xs = do
  first <- MaybeT [safeHead xs, safeIndex xs 1]  -- несколько кандидатов
  last' <- MaybeT [safeLast xs]
  return (first, last')

demo2 :: IO ()
demo2 = do
  let results = runMaybeT (findBothEnds [1,2,3,4,5])
  putStrLn $ "MaybeT над []: " ++ show results

demo2

In [ ]:
-- Пример 3: Парсер с MaybeT
-- Парсим строки настройки вида "key=value"
parseKeyValue :: String -> MaybeT IO (String, String)
parseKeyValue s = do
  let parts = break (== '=') s
  case parts of
    (k, '=':v) | not (null k) -> do
      liftIO $ putStrLn $ "Разобрано: " ++ show k ++ " = " ++ show v
      return (k, v)
    _ -> do
      liftIO $ putStrLn $ "Ошибка разбора: " ++ show s
      MaybeT (return Nothing)

testParser :: IO ()
testParser = do
  r1 <- runMaybeT $ parseKeyValue "host=localhost"
  r2 <- runMaybeT $ parseKeyValue "invalid-line"
  r3 <- runMaybeT $ parseKeyValue "port=8080"
  mapM_ print [r1, r2, r3]

testParser

## 5️⃣ ExceptT — обработка ошибок с информацией <a id="exceptt"></a>

```haskell
newtype ExceptT e m a = ExceptT { runExceptT :: m (Either e a) }
```

**ExceptT** — обобщение `MaybeT`: при провале несёт значение ошибки `e`.

### Ключевые функции
- `throwE :: e -> ExceptT e m a` — бросить ошибку
- `catchE :: ExceptT e m a -> (e -> ExceptT e' m a) -> ExceptT e' m a` — поймать ошибку
- `withExceptT :: (e -> e') -> ExceptT e m a -> ExceptT e' m a` — трансформировать ошибку

In [ ]:
import Control.Monad.Trans.Except

-- Тип ошибок для приложения
data AppError
  = NotFound String
  | ValidationError String
  | DatabaseError String
  deriving (Show, Eq)

type App a = ExceptT AppError IO a

-- Симуляция БД
type DB = Map String (Int, String)  -- имя -> (возраст, email)

fakeDB :: DB
fakeDB = Map.fromList
  [ ("alice", (30, "alice@example.com"))
  , ("bob",   (17, "bob@example.com"))
  ]

-- Операции с явными ошибками
lookupUser2 :: String -> App (Int, String)
lookupUser2 name = case Map.lookup name fakeDB of
  Nothing -> throwE (NotFound $ "Пользователь не найден: " ++ name)
  Just x  -> return x

validateAdult :: String -> Int -> App ()
validateAdult name age
  | age < 18  = throwE (ValidationError $ name ++ " моложе 18 лет (" ++ show age ++ ")")
  | otherwise = liftIO $ putStrLn $ "✓ " ++ name ++ " прошёл проверку возраста"

registerUser :: String -> App String
registerUser name = do
  (age, email) <- lookupUser2 name
  liftIO $ putStrLn $ "Найден: " ++ name ++ " <" ++ email ++ ">"
  validateAdult name age
  return $ "Зарегистрирован: " ++ name

testExcept :: IO ()
testExcept = do
  putStrLn "--- Alice (успех) ---"
  r1 <- runExceptT (registerUser "alice")
  print r1
  
  putStrLn "\n--- Bob (несовершеннолетний) ---"
  r2 <- runExceptT (registerUser "bob")
  print r2
  
  putStrLn "\n--- Zoro (не найден) ---"
  r3 <- runExceptT (registerUser "zoro")
  print r3

testExcept

In [ ]:
-- Пример: catchE и восстановление после ошибок
import Control.Monad.Trans.Except (withExceptT)

-- Трансформируем ошибку
dbLookup :: String -> ExceptT String IO Int
dbLookup "42" = return 42
dbLookup key  = throwE $ "Ключ не найден: " ++ key

-- Цепочка с восстановлением
withFallback :: String -> ExceptT String IO Int
withFallback key = dbLookup key `catchE` \err -> do
  liftIO $ putStrLn $ "Ошибка: " ++ err ++ " — используем значение по умолчанию"
  return 0

testCatch :: IO ()
testCatch = do
  r1 <- runExceptT (withFallback "42")
  putStrLn $ "Ключ '42': " ++ show r1
  r2 <- runExceptT (withFallback "99")
  putStrLn $ "Ключ '99': " ++ show r2

testCatch

In [ ]:
-- Пример: withExceptT — трансформация типа ошибки
data NetworkError = Timeout | ConnectionRefused deriving Show
data ServiceError = NetworkErr NetworkError | ParseErr String deriving Show

fetchData :: ExceptT NetworkError IO String
fetchData = throwE Timeout  -- симуляция

parseResponse :: String -> ExceptT String IO Int
parseResponse "42" = return 42
parseResponse s    = throwE $ "Не число: " ++ s

-- Объединяем в один тип ошибки
runService :: ExceptT ServiceError IO Int
runService = do
  raw <- withExceptT NetworkErr fetchData
  withExceptT ParseErr (parseResponse raw)

testService :: IO ()
testService = do
  r <- runExceptT runService
  print r

testService

## 6️⃣ StateT — состояние в стеке <a id="statet"></a>

```haskell
newtype StateT s m a = StateT { runStateT :: s -> m (a, s) }
```

**StateT** добавляет изменяемое состояние типа `s` поверх монады `m`.

### Ключевые функции
- `get :: StateT s m s` — прочитать состояние
- `put :: s -> StateT s m ()` — установить состояние
- `modify :: (s -> s) -> StateT s m ()` — изменить состояние
- `gets :: (s -> a) -> StateT s m a` — прочитать проекцию

### Категорная трактовка

`StateT s m` — это монада Клейсли для функтора `s -> m (-, s)`. Композиция стрелок Клейсли соответствует последовательному выполнению с передачей состояния.

In [ ]:
import Control.Monad.Trans.State

-- Пример 1: Счётчик с IO-логированием
type Counter = StateT Int IO

increment :: Counter ()
increment = do
  n <- get
  liftIO $ putStrLn $ "  Инкремент: " ++ show n ++ " -> " ++ show (n+1)
  put (n + 1)

addN :: Int -> Counter ()
addN n = modify (+n)

testCounter :: IO ()
testCounter = do
  (result, finalState) <- runStateT computation 0
  putStrLn $ "Итог: " ++ show result ++ ", состояние: " ++ show finalState
  where
    computation = do
      increment
      increment
      addN 10
      n <- get
      liftIO $ putStrLn $ "Текущее состояние: " ++ show n
      return (n * 2)

testCounter

In [ ]:
-- Пример 2: Интерпретатор простого стекового языка
data StackOp = Push Int | Pop | Add | Mul | Dup deriving (Show)

type Stack = [Int]
type StackM a = StateT Stack IO a

execOp :: StackOp -> StackM ()
execOp (Push n) = do
  modify (n:)
  s <- get
  liftIO $ putStrLn $ "PUSH " ++ show n ++ " -> " ++ show s
execOp Pop = do
  s <- get
  case s of
    []    -> liftIO $ putStrLn "ERROR: стек пуст!"
    (_:rest) -> do
      put rest
      liftIO $ putStrLn $ "POP -> " ++ show rest
execOp Add = do
  s <- get
  case s of
    (a:b:rest) -> do
      let result = a + b
      put (result:rest)
      liftIO $ putStrLn $ "ADD " ++ show a ++ "+" ++ show b ++ "=" ++ show result
    _ -> liftIO $ putStrLn "ERROR: недостаточно элементов"
execOp Mul = do
  s <- get
  case s of
    (a:b:rest) -> do
      let result = a * b
      put (result:rest)
      liftIO $ putStrLn $ "MUL " ++ show a ++ "*" ++ show b ++ "=" ++ show result
    _ -> liftIO $ putStrLn "ERROR: недостаточно элементов"
execOp Dup = do
  s <- get
  case s of
    []    -> liftIO $ putStrLn "ERROR: стек пуст!"
    (x:_) -> do
      modify (x:)
      liftIO $ putStrLn $ "DUP " ++ show x

runProgram :: [StackOp] -> IO Stack
runProgram ops = execStateT (mapM_ execOp ops) []

-- Вычислим (3 + 4) * 2
testStack :: IO ()
testStack = do
  putStrLn "=== Программа: (3+4)*2 ==="
  finalStack <- runProgram [Push 3, Push 4, Add, Push 2, Mul]
  putStrLn $ "Результат на стеке: " ++ show finalStack

testStack

In [ ]:
-- Пример 3: StateT для генерации уникальных ID
type GenM a = StateT Int IO a

newId :: GenM Int
newId = do
  i <- get
  put (i + 1)
  return i

data Expr2 = Lit2 Int | Add2 Expr2 Expr2 | Var2 String deriving Show
data AnnExpr = Ann { nodeId :: Int, node :: AnnNode }
data AnnNode = ALit Int | AAdd AnnExpr AnnExpr | AVar String
instance Show AnnExpr where
  show (Ann i n) = "[" ++ show i ++ "]" ++ showNode n
    where
      showNode (ALit x) = show x
      showNode (AAdd l r) = "(" ++ show l ++ "+" ++ show r ++ ")"
      showNode (AVar s) = s

-- Аннотируем дерево уникальными ID
annotate :: Expr2 -> GenM AnnExpr
annotate (Lit2 n) = do
  i <- newId
  return (Ann i (ALit n))
annotate (Var2 s) = do
  i <- newId
  return (Ann i (AVar s))
annotate (Add2 l r) = do
  al <- annotate l
  ar <- annotate r
  i  <- newId
  return (Ann i (AAdd al ar))

testAnnotate :: IO ()
testAnnotate = do
  let expr = Add2 (Lit2 1) (Add2 (Var2 "x") (Lit2 2))
  (ann, count) <- runStateT (annotate expr) 0
  putStrLn $ "Аннотированное дерево: " ++ show ann
  putStrLn $ "Всего узлов: " ++ show count

testAnnotate

## 7️⃣ ReaderT — окружение в стеке <a id="readert"></a>

```haskell
newtype ReaderT r m a = ReaderT { runReaderT :: r -> m a }
```

**ReaderT** добавляет доступ к неизменяемому окружению `r`.

### Ключевые функции
- `ask :: ReaderT r m r` — получить всё окружение
- `asks :: (r -> a) -> ReaderT r m a` — получить проекцию
- `local :: (r -> r) -> ReaderT r m a -> ReaderT r m a` — локально изменить окружение

### Категорная трактовка

`ReaderT r m` — это монада функций `r -> m a`. Это **экспоненциал** в категории функторов. `local` реализует **ко-монадическое** действие: изменяет «контекст» без изменения значения.

`Reader r` изоморфен `(->) r`, который является **ко-монадой** (Traced). `ReaderT r m` сочетает оба аспекта.

In [ ]:
import Control.Monad.Trans.Reader

-- Конфигурация приложения
data Config = Config
  { cfgHost    :: String
  , cfgPort    :: Int
  , cfgDebug   :: Bool
  , cfgMaxConn :: Int
  } deriving Show

defaultConfig :: Config
defaultConfig = Config
  { cfgHost    = "localhost"
  , cfgPort    = 8080
  , cfgDebug   = True
  , cfgMaxConn = 100
  }

type AppM a = ReaderT Config IO a

-- Логирование, которое смотрит на флаг debug
logMsg :: String -> AppM ()
logMsg msg = do
  debug <- asks cfgDebug
  if debug
    then liftIO $ putStrLn $ "[DEBUG] " ++ msg
    else return ()

-- Подключение к БД с конфигом из Reader
connectDB :: AppM String
connectDB = do
  host <- asks cfgHost
  port <- asks cfgPort
  logMsg $ "Подключаемся к " ++ host ++ ":" ++ show port
  return $ "Connection{" ++ host ++ ":" ++ show port ++ "}"

-- Основная логика
runApp :: AppM ()
runApp = do
  conn <- connectDB
  logMsg $ "Соединение: " ++ conn
  maxConn <- asks cfgMaxConn
  liftIO $ putStrLn $ "Максимум соединений: " ++ show maxConn

testReader :: IO ()
testReader = do
  putStrLn "=== Обычная конфигурация ==="
  runReaderT runApp defaultConfig
  
  putStrLn "\n=== Production (debug=False) ==="
  let prodConfig = defaultConfig { cfgDebug = False, cfgHost = "prod.example.com" }
  runReaderT runApp prodConfig

testReader

In [ ]:
-- Пример: local для временного изменения окружения
type Env = Map String Int  -- переменные

type EvalM a = ReaderT Env IO a

evalVar :: String -> EvalM Int
evalVar name = do
  env <- ask
  case Map.lookup name env of
    Just v  -> do
      liftIO $ putStrLn $ "  " ++ name ++ " = " ++ show v
      return v
    Nothing -> do
      liftIO $ putStrLn $ "  " ++ name ++ " не определена!"
      return 0

-- let x = 10 in (let y = 20 in x + y) + x
evalExprDemo :: EvalM Int
evalExprDemo = do
  x <- evalVar "x"   -- из окружения
  inner <- local (Map.insert "y" 20) $ do
    -- здесь "y" определена локально
    y <- evalVar "y"
    return (x + y)
  return (inner + x)

testLocal :: IO ()
testLocal = do
  let env = Map.fromList [("x", 10), ("z", 99)]
  result <- runReaderT evalExprDemo env
  putStrLn $ "Результат: " ++ show result

testLocal

## 8️⃣ WriterT — журналирование в стеке <a id="writert"></a>

```haskell
newtype WriterT w m a = WriterT { runWriterT :: m (a, w) }
```

где `w` — **моноид** (нужен для `mempty` и `mappend`).

**WriterT** накапливает «побочный выход» `w` (лог, список, etc.).

### Ключевые функции
- `tell :: w -> WriterT w m ()` — добавить в лог
- `listen :: WriterT w m a -> WriterT w m (a, w)` — получить лог вместе с результатом
- `pass :: WriterT w m (a, w -> w) -> WriterT w m a` — модифицировать лог

### Осторожно!
Стандартный `WriterT` со списком имеет проблему с производительностью из-за `(++)`. Для логов лучше использовать `Data.Sequence` или `DList`.

In [ ]:
import Control.Monad.Trans.Writer

-- Пример 1: Трассировка вычислений
type Traced a = WriterT [String] IO a

-- Факториал с трассировкой
factTraced :: Int -> Traced Int
factTraced 0 = do
  tell ["fact(0) = 1"]
  return 1
factTraced n = do
  tell ["fact(" ++ show n ++ "): рекурсия..."]
  sub <- factTraced (n - 1)
  let result = n * sub
  tell ["fact(" ++ show n ++ ") = " ++ show result]
  return result

testWriter :: IO ()
testWriter = do
  (result, logs) <- runWriterT (factTraced 4)
  putStrLn $ "Результат: " ++ show result
  putStrLn "\nЛог выполнения:"
  mapM_ putStrLn logs

testWriter

In [ ]:
-- Пример 2: Сбор предупреждений при компиляции
data Warning = Warning { warnLine :: Int, warnMsg :: String } deriving Show

type CompileM a = WriterT [Warning] IO a

warn :: Int -> String -> CompileM ()
warn line msg = tell [Warning line msg]

data SimpleType = TInt | TBool | TStr deriving (Show, Eq)

inferType :: String -> CompileM SimpleType
inferType "true"  = return TBool
inferType "false" = return TBool
inferType s
  | all (`elem` "0123456789") s = return TInt
  | otherwise = do
      warn 1 $ "Неизвестный литерал '" ++ s ++ "', предполагаем Str"
      return TStr

checkTypes :: [String] -> CompileM [SimpleType]
checkTypes exprs = do
  types <- mapM inferType exprs
  let pairs = zip exprs types
  mapM_ (\(e,t) -> liftIO $ putStrLn $ "  " ++ e ++ " :: " ++ show t) pairs
  let hasInt = TInt `elem` types
      hasStr = TStr `elem` types
  if hasInt && hasStr
    then warn 1 "Смешиваются Int и Str в одном выражении"
    else return ()
  return types

testCompile :: IO ()
testCompile = do
  let exprs = ["42", "true", "hello", "123", "world"]
  (types, warnings) <- runWriterT (checkTypes exprs)
  putStrLn $ "\nПредупреждения (" ++ show (length warnings) ++ "):"
  mapM_ print warnings

testCompile

## 9️⃣ ContT — продолжения <a id="contt"></a>

```haskell
newtype ContT r m a = ContT { runContT :: (a -> m r) -> m r }
```

**ContT** — самый мощный и самый сложный трансформер. Представляет вычисление как функцию, принимающую **продолжение** (continuation) — что делать с результатом.

### Ключевые функции
- `callCC :: ((a -> ContT r m b) -> ContT r m a) -> ContT r m a` — захват текущего продолжения
- `cont :: ((a -> r) -> r) -> ContT r m a` — создать ContT из CPS-функции
- `shiftT`, `resetT` — ограниченные продолжения

### Категорная трактовка

`ContT r m` реализует **монаду продолжений** в CPS (continuation-passing style). Это **монада двойного отрицания**: `a ≅ ((a -> r) -> r)` при `r = ⊥`. `callCC` реализует принцип **Peirce**: `((A → B) → A) → A`.

In [ ]:
import Control.Monad.Trans.Cont

-- Пример 1: ранний выход через callCC
searchList :: [Int] -> Int -> ContT Bool IO Bool
searchList xs target = callCC $ \exit -> do
  mapM_ (\x -> do
    liftIO $ putStrLn $ "  Проверяем: " ++ show x
    if x == target
      then do
        liftIO $ putStrLn $ "  Нашли! Ранний выход."
        exit True  -- немедленный выход
      else return ()
    ) xs
  return False

testCont :: IO ()
testCont = do
  putStrLn "Ищем 3 в [1,2,3,4,5]:"
  r1 <- runContT (searchList [1,2,3,4,5] 3) return
  putStrLn $ "Найдено: " ++ show r1
  putStrLn "\nИщем 9 в [1,2,3]:"
  r2 <- runContT (searchList [1,2,3] 9) return
  putStrLn $ "Найдено: " ++ show r2

testCont

In [ ]:
-- Пример 2: ContT как обобщённый break/continue
-- Вложенные циклы с break
type LoopM = ContT () IO

loop :: [a] -> (a -> LoopM ()) -> LoopM ()
loop xs body = mapM_ body xs

breakWhen :: Bool -> ContT () IO () -> ContT () IO ()
breakWhen cond k = if cond then return () else k

-- Поиск пары (i,j) где i*j == 12
findPair :: IO ()
findPair = runContT go return
  where
    go :: LoopM ()
    go = callCC $ \breakOuter -> do
      loop [1..5] $ \i -> do
        callCC $ \breakInner -> do
          loop [1..5] $ \j -> do
            liftIO $ putStr $ "(" ++ show i ++ "," ++ show j ++ ") "
            if i * j == 12
              then do
                liftIO $ putStrLn $ "\nНашли: " ++ show i ++ "*" ++ show j ++ "=12"
                breakOuter ()  -- выход из обоих циклов!
              else return ()

findPair

## 🔟 Интересные комбинации: стеки трансформеров <a id="stacks"></a>

### Стандартный стек: ReaderT + StateT + ExceptT

Наиболее распространённый паттерн в реальных приложениях:

```haskell
type App e s r a = ReaderT r (StateT s (ExceptT e IO)) a
```

Читается снизу вверх:
- `IO` — базовая монада
- `ExceptT e` — обработка ошибок
- `StateT s` — изменяемое состояние
- `ReaderT r` — конфигурация

### Порядок трансформеров имеет значение!

| Порядок | Семантика |
|---------|----------|
| `StateT s (Either e)` | При ошибке состояние **теряется** |
| `ExceptT e (State s)` | При ошибке состояние **сохраняется** |
| `MaybeT []` | Список Maybe: отказ убирает ветку |
| `WriterT w Maybe` | Лог есть только при успехе |
| `MaybeT (Writer w)` | Лог накапливается даже при отказе |

In [ ]:
{-# LANGUAGE GeneralizedNewtypeDeriving #-}
import Control.Monad.Trans.Except
import Control.Monad.Trans.State
import Control.Monad.Trans.Reader

-- Демонстрация: порядок трансформеров имеет значение

-- Вариант A: ExceptT внешний, State внутренний
-- При ошибке состояние СОХРАНЯЕТСЯ
type AppA s e a = ExceptT e (StateT s IO) a

runAppA :: AppA s e a -> s -> IO (Either e a, s)
runAppA m s = runStateT (runExceptT m) s

demoA :: AppA Int String String
demoA = do
  lift $ modify (+1)  -- состояние = 1
  lift $ modify (+1)  -- состояние = 2
  throwE "Ошибка!"    -- бросаем исключение
  lift $ modify (+1)  -- не выполнится
  return "OK"

-- Вариант B: State внешний, ExceptT внутренний
-- При ошибке состояние ТЕРЯЕТСЯ (откатывается)
type AppB s e a = StateT s (ExceptT e IO) a

runAppB :: AppB s e a -> s -> IO (Either e (a, s))
runAppB m s = runExceptT (runStateT m s)

demoB :: AppB Int String String
demoB = do
  modify (+1)  -- состояние = 1
  modify (+1)  -- состояние = 2
  lift $ throwE "Ошибка!"
  modify (+1)
  return "OK"

testOrder :: IO ()
testOrder = do
  putStrLn "=== ExceptT (StateT IO) — состояние сохраняется ==="
  (resultA, stateA) <- runAppA demoA 0
  putStrLn $ "  Результат: " ++ show resultA
  putStrLn $ "  Состояние: " ++ show stateA ++ " (сохранено!)"

  putStrLn "\n=== StateT (ExceptT IO) — состояние теряется ==="
  resultB <- runAppB demoB 0
  putStrLn $ "  Результат: " ++ show resultB ++ " (состояние потеряно)"

testOrder

### Полный стек приложения: Reader + State + Except + IO

Пример реального приложения — мини-интерпретатор с конфигурацией, состоянием и обработкой ошибок.

In [ ]:
-- Полноценный стек приложения
data AppConfig = AppConfig
  { maxSteps :: Int
  , verbose  :: Bool
  } deriving Show

data AppState = AppState
  { stepCount :: Int
  , variables :: Map String Int
  } deriving Show

data AppErr
  = UndefinedVar String
  | StepLimitExceeded
  | DivByZero
  deriving Show

-- Стек: ReaderT Config (StateT AppState (ExceptT AppErr IO))
type AppStack a = ReaderT AppConfig (StateT AppState (ExceptT AppErr IO)) a

runAppStack :: AppConfig -> AppState -> AppStack a -> IO (Either AppErr (a, AppState))
runAppStack cfg st m = runExceptT (runStateT (runReaderT m cfg) st)

-- Вспомогательные функции
step :: AppStack ()
step = do
  cfg <- ask
  st  <- lift get
  let n = stepCount st
  if n >= maxSteps cfg
    then lift (lift (throwE StepLimitExceeded))
    else do
      lift (put st { stepCount = n + 1 })
      if verbose cfg
        then liftIO $ putStrLn $ "  Шаг " ++ show (n+1)
        else return ()

setVar :: String -> Int -> AppStack ()
setVar name val = do
  lift $ modify $ \s -> s { variables = Map.insert name val (variables s) }

getVar :: String -> AppStack Int
getVar name = do
  vars <- lift $ gets variables
  case Map.lookup name vars of
    Just v  -> return v
    Nothing -> lift (lift (throwE (UndefinedVar name)))

safeDiv2 :: Int -> Int -> AppStack Int
safeDiv2 _ 0 = lift (lift (throwE DivByZero))
safeDiv2 x y = return (x `div` y)

-- Программа: x=10, y=3, z=x/y, w=x+y+z
program :: AppStack Int
program = do
  step; setVar "x" 10
  step; setVar "y" 3
  step; x <- getVar "x"; y <- getVar "y"
  step; z <- safeDiv2 x y; setVar "z" z
  step; w <- return (x + y + z); setVar "w" w
  return w

testAppStack :: IO ()
testAppStack = do
  let cfg = AppConfig { maxSteps = 10, verbose = True }
      st0 = AppState { stepCount = 0, variables = Map.empty }
  putStrLn "=== Выполнение программы ==="
  result <- runAppStack cfg st0 program
  case result of
    Left err -> putStrLn $ "Ошибка: " ++ show err
    Right (val, finalSt) -> do
      putStrLn $ "Результат: " ++ show val
      putStrLn $ "Шагов: " ++ show (stepCount finalSt)
      putStrLn $ "Переменные: " ++ show (variables finalSt)

testAppStack

### Другие интересные комбинации

#### MaybeT [] — недетерминизм с отказом

Комбинация списка (недетерминизм) и Maybe (отказ). Позволяет «убивать» ветки вычисления.

In [ ]:
-- MaybeT [] — недетерминизм + отказ
-- Тип: MaybeT [] a = [Maybe a]
-- Nothing = убрать эту ветку

-- Задача: найти все пары (x,y) из списка где x < y и x+y == target
pairsWithSum :: [Int] -> Int -> [(Int, Int)]
pairsWithSum xs target = do
  -- Это список-монада (недетерминизм)
  x <- xs
  y <- xs
  if x < y && x + y == target then return (x, y) else []

-- То же с MaybeT
pairsWithSumMT :: [Int] -> Int -> MaybeT [] (Int, Int)
pairsWithSumMT xs target = do
  x <- lift xs
  y <- lift xs
  MaybeT [if x < y && x + y == target then Just (x,y) else Nothing]

testMaybeList :: IO ()
testMaybeList = do
  let xs = [1..5]
  putStrLn $ "Пары с суммой 6: " ++ show (pairsWithSum xs 6)
  let mt = runMaybeT (pairsWithSumMT xs 6)
  putStrLn $ "MaybeT [] сырой: " ++ show mt
  putStrLn $ "MaybeT [] отфильтрованный: " ++ show [x | Just x <- mt]

testMaybeList

#### WriterT w Maybe vs MaybeT (Writer w)

Порядок влияет на то, накапливается ли лог при отказе.

In [ ]:
import Data.Functor.Identity (Identity(..))
import Control.Monad.Trans.Writer (runWriterT)

-- WriterT [String] Maybe: лог есть только при успехе
-- MaybeT (Writer [String]): лог накапливается даже при отказе

-- Вариант А: WriterT внешний
compA :: WriterT [String] Maybe Int
compA = do
  tell ["Шаг 1"]
  tell ["Шаг 2"]
  lift Nothing  -- отказ
  tell ["Шаг 3"]
  return 42

-- Вариант B: MaybeT внешний  
compB :: MaybeT (WriterT [String] Identity) Int
compB = do
  lift $ tell ["Шаг 1"]
  lift $ tell ["Шаг 2"]
  MaybeT (WriterT (Identity (Nothing, ["Отказ"])))
  lift $ tell ["Шаг 3"]
  return 42

testWriterMaybe :: IO ()
testWriterMaybe = do
  putStrLn "=== WriterT [] Maybe (лог теряется при отказе) ==="
  print (runWriterT compA)

  putStrLn "=== MaybeT (WriterT []) (лог сохраняется при отказе) ==="
  let Identity (result, logs) = runWriterT (runMaybeT compB)
  putStrLn $ "Результат: " ++ show result
  putStrLn $ "Лог: " ++ show logs

testWriterMaybe

#### StateT s [] — недетерминированное состояние

`StateT s []` даёт список всех возможных пар `(значение, состояние)` — мощный инструмент для поиска с состоянием.

In [ ]:
-- StateT Int [] — все пути с состоянием
-- Задача: N-ферзей — размещение ферзей на доске N×N

type BoardM a = StateT [Int] [] a  -- состояние = список занятых столбцов по строкам

isSafe :: [Int] -> Int -> Bool
isSafe placed col = all (\(row, c) -> c /= col && abs (c - col) /= length placed - row)
                        (zip [0..] placed)

-- Недетерминированно выбираем безопасный столбец
placeQueen :: Int -> StateT [Int] [] ()
placeQueen n = do
  placed <- get
  col <- lift [0..n-1]  -- выбираем из всех столбцов (недетерминизм)
  if isSafe placed col
    then put (placed ++ [col])
    else lift []  -- неудача: эта ветка отмирает

solveNQueens :: Int -> [[Int]]
solveNQueens n = map snd $ runStateT (mapM_ (const (placeQueen n)) [1..n]) []

testQueens :: IO ()
testQueens = do
  let n = 6
      solutions = solveNQueens n
  putStrLn $ "N-ферзей для N=" ++ show n ++ ":"
  putStrLn $ "Всего решений: " ++ show (length solutions)
  putStrLn "Первые 3 решения (столбец ферзя в каждой строке):"
  mapM_ print (take 3 solutions)

testQueens

## 11. mtl-стиль: классы для стеков <a id="mtl"></a>

**Проблема** с явными `lift`: при добавлении нового слоя нужно переписывать все `lift`.

**Решение** в mtl: классы типов, которые автоматически «пробрасывают» операции через слои.

```haskell
class Monad m => MonadState s m | m -> s where
  get :: m s
  put :: s -> m ()
```

Экземпляры: `State s`, `StateT s m` (если `Monad m`), и **автоматически** `ReaderT r (StateT s m)` и т.д.

### Закон сопряжения (adjunction)

`StateT s` сопряжён с некоторым коэндофунктором. Классы mtl реализуют **свободные теоремы** для этих сопряжений:

`MonadState s m` = «существует морфизм монад из `State s` в `m`»

In [ ]:
import Control.Monad.IO.Class (MonadIO, liftIO)

{-# LANGUAGE MultiParamTypeClasses #-}
{-# LANGUAGE FlexibleInstances #-}
import qualified Control.Monad.State.Class as SC
import qualified Control.Monad.Reader.Class as RC
import qualified Control.Monad.Error.Class as EC

-- С mtl-классами не нужны явные lift!
-- Функции используют классовые ограничения

-- Без mtl (с явными lift):
setVarExplicit :: String -> Int -> ReaderT r (StateT (Map String Int) IO) ()
setVarExplicit k v = lift $ modify (Map.insert k v)

-- С mtl (полиморфная функция):
setVarMtl :: (SC.MonadState (Map String Int) m) => String -> Int -> m ()
setVarMtl k v = SC.modify (Map.insert k v)

-- Теперь setVarMtl работает в ЛЮБОМ стеке, содержащем State!

logMtl :: (RC.MonadReader Bool m, MonadIO m) => String -> m ()
logMtl msg = do
  debug <- RC.ask
  if debug then liftIO (putStrLn $ "[LOG] " ++ msg) else return ()

-- Функция работает в любом стеке с Reader Bool и IO
doWorkMtl :: (SC.MonadState (Map String Int) m,
              RC.MonadReader Bool m,
              MonadIO m) => m ()
doWorkMtl = do
  logMtl "Начало работы"
  setVarMtl "x" 42
  setVarMtl "y" 100
  logMtl "Работа выполнена"

testMtl :: IO ()
testMtl = do
  let run = runStateT (runReaderT doWorkMtl True) Map.empty
  (_, finalState) <- run
  putStrLn $ "Состояние: " ++ show finalState

testMtl

## 12. Реальные примеры <a id="реальные-примеры"></a>

### 12.1 Простой монадический парсер

Парсер — это классический пример трансформеров: `StateT String (ExceptT String IO)`.

In [ ]:
-- Монадический парсер на трансформерах
import Data.Char (isAlpha, isDigit, isSpace)

type ParseError = String
type Input = String

type Parser a = StateT Input (ExceptT ParseError IO) a

runParser :: Parser a -> Input -> IO (Either ParseError (a, Input))
runParser p input = runExceptT (runStateT p input)

-- Примитивы
satisfy :: (Char -> Bool) -> String -> Parser Char
satisfy pred desc = do
  input <- get
  case input of
    [] -> lift $ throwE $ "Ожидался " ++ desc ++ ", но достигнут конец строки"
    (c:rest)
      | pred c    -> put rest >> return c
      | otherwise -> lift $ throwE $ "Ожидался " ++ desc ++ ", встретил '" ++ [c] ++ "'"

char :: Char -> Parser Char
char c = satisfy (== c) ("'" ++ [c] ++ "'")

digit :: Parser Char
digit = satisfy isDigit "цифру"

letter :: Parser Char
letter = satisfy isAlpha "букву"

spaces :: Parser ()
spaces = do
  input <- get
  let rest = dropWhile isSpace input
  put rest

-- Парсер числа
number :: Parser Int
number = do
  spaces
  digits <- some digit
  return (read digits)
  where
    some p = do
      c  <- p
      cs <- many p
      return (c:cs)
    many p = do
      input <- get
      case input of
        []    -> return []
        (c:_) -> if isDigit c
                 then do { x <- p; xs <- many p; return (x:xs) }
                 else return []

-- Парсер суммы: number '+' number
parseSum :: Parser Int
parseSum = do
  a <- number
  spaces
  char '+'
  b <- number
  return (a + b)

testParser2 :: IO ()
testParser2 = do
  let tests = ["123 + 456", "42 + 0", "abc", "10 +"]
  mapM_ (\s -> do
    r <- runParser parseSum s
    putStrLn $ show s ++ " => " ++ show r
    ) tests

testParser2

### 12.2 Интерпретатор с полным стеком

Интерпретатор простого императивного языка.

In [ ]:
-- Простой императивный язык
data Stmt
  = Assign String AExpr
  | If BExpr [Stmt] [Stmt]
  | While BExpr [Stmt]
  | Print AExpr
  deriving Show

data AExpr
  = Num Int
  | Var String
  | BinOp AExpr Char AExpr
  deriving Show

data BExpr
  = Less AExpr AExpr
  | Equal AExpr AExpr
  | Not BExpr
  deriving Show

-- Стек интерпретатора
type InterpState = Map String Int
data InterpConfig = InterpConfig { maxIterations :: Int } deriving Show
data InterpError = UndefVariable String | MaxIter | DivBy0 deriving Show

type Interp a = ReaderT InterpConfig (StateT InterpState (ExceptT InterpError IO)) a

runInterp :: InterpConfig -> InterpState -> Interp a -> IO (Either InterpError (a, InterpState))
runInterp cfg st m = runExceptT $ runStateT (runReaderT m cfg) st

evalA :: AExpr -> Interp Int
evalA (Num n) = return n
evalA (Var s) = do
  vars <- lift get
  case Map.lookup s vars of
    Just v  -> return v
    Nothing -> lift (lift (throwE (UndefVariable s)))
evalA (BinOp l '+' r) = (+) <$> evalA l <*> evalA r
evalA (BinOp l '-' r) = (-) <$> evalA l <*> evalA r
evalA (BinOp l '*' r) = (*) <$> evalA l <*> evalA r
evalA (BinOp l '/' r) = do
  rv <- evalA r
  if rv == 0 then lift (lift (throwE DivBy0))
  else (`div` rv) <$> evalA l
evalA _ = return 0

evalB :: BExpr -> Interp Bool
evalB (Less a b)  = (<)  <$> evalA a <*> evalA b
evalB (Equal a b) = (==) <$> evalA a <*> evalA b
evalB (Not b)     = not  <$> evalB b

execStmt :: Int -> Stmt -> Interp ()
execStmt _     (Assign v e) = do
  val <- evalA e
  lift $ modify (Map.insert v val)
execStmt _     (Print e) = do
  val <- evalA e
  liftIO $ putStrLn $ ">>> " ++ show val
execStmt depth (If cond t f) = do
  c <- evalB cond
  mapM_ (execStmt depth) (if c then t else f)
execStmt depth (While cond body) = do
  maxI <- asks maxIterations
  go maxI
  where
    go 0 = lift (lift (throwE MaxIter))
    go n = do
      c <- evalB cond
      if c then do
        mapM_ (execStmt depth) body
        go (n-1)
      else return ()

-- Программа: факториал через while
-- n = 5; acc = 1; while n > 0: acc = acc*n; n = n-1; print acc
factProgram :: [Stmt]
factProgram =
  [ Assign "n"   (Num 5)
  , Assign "acc" (Num 1)
  , While (Not (Equal (Var "n") (Num 0)))
      [ Assign "acc" (BinOp (Var "acc") '*' (Var "n"))
      , Assign "n"   (BinOp (Var "n")   '-' (Num 1))
      ]
  , Print (Var "acc")
  ]

testInterp :: IO ()
testInterp = do
  let cfg = InterpConfig { maxIterations = 1000 }
  putStrLn "=== Вычисляем 5! ==="
  r <- runInterp cfg Map.empty (mapM_ (execStmt 0) factProgram)
  case r of
    Left err -> putStrLn $ "Ошибка: " ++ show err
    Right (_, st) -> putStrLn $ "Финальное состояние: " ++ show st

testInterp

### 12.3 Web-handler <a id="web-handler"></a>

Мини web-сервер: `ReaderT Config (ExceptT AppError IO)`.
Каждый запрос читает конфигурацию, может вернуть ошибку и выполняет IO.


In [ ]:
import Control.Monad.Trans.Class (lift)
import Control.Monad.Trans.Reader
import Control.Monad.Trans.Except (ExceptT, throwE, runExceptT)
import Data.Map.Strict (Map)
import qualified Data.Map.Strict as Map

-- Стек: ReaderT Config (ExceptT AppError IO)
type WebM a = ReaderT Config (ExceptT AppError IO) a

data Config = Config
  { cfgHost    :: String
  , cfgPort    :: Int
  , cfgRoutes  :: Map String (WebM String)
  }

data AppError
  = NotFound String
  | Unauthorized
  | InternalError String
  deriving Show

runWebM :: Config -> WebM a -> IO (Either AppError a)
runWebM cfg action = runExceptT (runReaderT action cfg)

-- Вспомогательные функции
getConfig :: WebM Config
getConfig = ask

throwApp :: AppError -> WebM a
throwApp = lift . throwE

liftWebIO :: IO a -> WebM a
liftWebIO = lift . lift

-- Обработчики маршрутов
homeHandler :: WebM String
homeHandler = do
  cfg <- getConfig
  liftWebIO $ putStrLn $ "[GET /] host=" ++ cfgHost cfg
  return "<h1>Добро пожаловать!</h1>"

usersHandler :: String -> WebM String
usersHandler userId = do
  liftWebIO $ putStrLn $ "[GET /users/" ++ userId ++ "]"
  if userId == "admin"
    then throwApp Unauthorized
    else return $ "<p>Пользователь: " ++ userId ++ "</p>"

notFoundHandler :: String -> WebM String
notFoundHandler path = throwApp (NotFound path)

-- Диспетчер запросов
dispatch :: String -> WebM String
dispatch path = do
  cfg <- getConfig
  case Map.lookup path (cfgRoutes cfg) of
    Just handler -> handler
    Nothing      -> notFoundHandler path

-- Симуляция нескольких запросов
testServer :: IO ()
testServer = do
  let routes = Map.fromList
        [ ("/",           homeHandler)
        , ("/users/bob",  usersHandler "bob")
        , ("/users/admin", usersHandler "admin")
        ]
      cfg = Config { cfgHost = "localhost", cfgPort = 8080, cfgRoutes = routes }
      requests = ["/", "/users/bob", "/users/admin", "/unknown"]
  mapM_ (\path -> do
    result <- runWebM cfg (dispatch path)
    case result of
      Right body -> putStrLn $ path ++ " -> OK: " ++ take 40 body
      Left err   -> putStrLn $ path ++ " -> ERR: " ++ show err
    ) requests

testServer


## Итоговая таблица трансформеров

| Трансформер | Тип | Добавляет | `run` функция |
|-------------|-----|-----------|---------------|
| `MaybeT m` | `m (Maybe a)` | Отказ без информации | `runMaybeT` |
| `ExceptT e m` | `m (Either e a)` | Ошибка с типом `e` | `runExceptT` |
| `StateT s m` | `s -> m (a, s)` | Изменяемое состояние | `runStateT` / `evalStateT` / `execStateT` |
| `ReaderT r m` | `r -> m a` | Окружение только для чтения | `runReaderT` |
| `WriterT w m` | `m (a, w)` | Накапливаемый вывод (моноид) | `runWriterT` |
| `ContT r m` | `(a -> m r) -> m r` | Продолжения / нелинейный поток | `runContT` |

### Советы по применению

**Порядок в стеке** (снизу вверх — от базового к внешнему):
```
ReaderT Config       -- конфигурация (внешний)
StateT AppState      -- состояние
ExceptT AppError     -- ошибки
WriterT [Log]        -- логирование
IO                   -- базовая монада (внутренняя)
```

**Правила**:
1. `IO` всегда снизу (или используйте `liftIO`)
2. `ExceptT` снаружи `StateT` — состояние сохраняется при ошибке
3. `StateT` снаружи `ExceptT` — состояние откатывается при ошибке
4. Используйте mtl-классы (`MonadState`, `MonadReader`, etc.) для переносимого кода
5. Для производительности: `Data.Sequence` вместо `[a]` в `WriterT`

## Диаграмма стека трансформеров





In [ ]:
-- Diagram: monad transformer stack
stackSvg :: String
stackSvg = unlines
  [ "<svg xmlns=\"http://www.w3.org/2000/svg\" width=\"400\" height=\"380\" viewBox=\"0 0 400 380\" font-family=\"monospace,Arial,sans-serif\">"
  , "  <rect width=\"400\" height=\"380\" fill=\"#ffffff\" rx=\"8\"/>"
  , "  <text x=\"200\" y=\"28\" text-anchor=\"middle\" font-size=\"14\" font-weight=\"bold\" fill=\"#334155\">Monad Transformer Stack</text>"
  , "  <rect x=\"20\" y=\"45\" width=\"360\" height=\"50\" fill=\"#dbeafe\" stroke=\"#1d4ed8\" stroke-width=\"2\" rx=\"6\"/>"
  , "  <text x=\"200\" y=\"68\" text-anchor=\"middle\" font-size=\"12\" font-weight=\"bold\" fill=\"#1e40af\">ReaderT Config</text>"
  , "  <text x=\"200\" y=\"85\" text-anchor=\"middle\" font-size=\"10\" fill=\"#64748b\">ask, asks, local</text>"
  , "  <rect x=\"35\" y=\"105\" width=\"330\" height=\"50\" fill=\"#d1fae5\" stroke=\"#34d399\" stroke-width=\"2\" rx=\"6\"/>"
  , "  <text x=\"200\" y=\"128\" text-anchor=\"middle\" font-size=\"12\" font-weight=\"bold\" fill=\"#065f46\">StateT AppState</text>"
  , "  <text x=\"200\" y=\"145\" text-anchor=\"middle\" font-size=\"10\" fill=\"#64748b\">get, put, modify</text>"
  , "  <rect x=\"50\" y=\"165\" width=\"300\" height=\"50\" fill=\"#fef3c7\" stroke=\"#f59e0b\" stroke-width=\"2\" rx=\"6\"/>"
  , "  <text x=\"200\" y=\"188\" text-anchor=\"middle\" font-size=\"12\" font-weight=\"bold\" fill=\"#92400e\">WriterT [Log]</text>"
  , "  <text x=\"200\" y=\"205\" text-anchor=\"middle\" font-size=\"10\" fill=\"#64748b\">tell, listen, pass</text>"
  , "  <rect x=\"65\" y=\"225\" width=\"270\" height=\"50\" fill=\"#fee2e2\" stroke=\"#ef4444\" stroke-width=\"2\" rx=\"6\"/>"
  , "  <text x=\"200\" y=\"248\" text-anchor=\"middle\" font-size=\"12\" font-weight=\"bold\" fill=\"#991b1b\">ExceptT AppError</text>"
  , "  <text x=\"200\" y=\"265\" text-anchor=\"middle\" font-size=\"10\" fill=\"#64748b\">throwE, catchE</text>"
  , "  <rect x=\"80\" y=\"285\" width=\"240\" height=\"50\" fill=\"#ede9fe\" stroke=\"#8b5cf6\" stroke-width=\"2\" rx=\"6\"/>"
  , "  <text x=\"200\" y=\"308\" text-anchor=\"middle\" font-size=\"12\" font-weight=\"bold\" fill=\"#5b21b6\">IO</text>"
  , "  <text x=\"200\" y=\"325\" text-anchor=\"middle\" font-size=\"10\" fill=\"#64748b\">liftIO, base effects</text>"
  , "  <text x=\"10\" y=\"80\"  font-size=\"9\" fill=\"#64748b\">lift</text>"
  , "  <text x=\"10\" y=\"140\" font-size=\"9\" fill=\"#64748b\">lift</text>"
  , "  <text x=\"10\" y=\"200\" font-size=\"9\" fill=\"#64748b\">lift</text>"
  , "  <text x=\"10\" y=\"260\" font-size=\"9\" fill=\"#64748b\">liftIO</text>"
  , "  <text x=\"200\" y=\"365\" text-anchor=\"middle\" font-size=\"9\" fill=\"#64748b\">run: runReaderT . runStateT . runWriterT . runExceptT</text>"
  , "</svg>"
  ]

writeFile "/tmp/mt_stack.svg" stackSvg
putStrLn "Stack diagram saved."

In [ ]:
:! cp /tmp/mt_stack.svg /home/jovyan/src/mt_stack.svg && echo OK

## Бонус: Свободные монады и трансформеры

Свободная монада `Free f a` — альтернативный подход к эффектам:

```haskell
data Free f a = Pure a | Free (f (Free f a))
```

В отличие от трансформеров:
- `Free f` позволяет **интерпретировать** программу несколькими способами
- Трансформеры **фиксируют** один способ интерпретации
- `Free` мощнее но медленнее; трансформеры быстрее

Связь: любая монада `m` порождает пару сопряжённых функторов, и `Free` — левое сопряжение к «забывающему» функтору.

Библиотека `freer-simple` реализует **свободные эффекты** без вложенных `newtype`.

In [ ]:
-- Демонстрация свободной монады вручную
data TelType next
  = GetLine (String -> next)
  | PutLine String next
  deriving Functor

data Free f a = Pure a | Wrap (f (Free f a))

instance Functor f => Functor (Free f) where
  fmap f (Pure a) = Pure (f a)
  fmap f (Wrap x) = Wrap (fmap (fmap f) x)

instance Functor f => Applicative (Free f) where
  pure = Pure
  (Pure f)  <*> x = fmap f x
  (Wrap ff) <*> x = Wrap (fmap (<*> x) ff)

instance Functor f => Monad (Free f) where
  return = Pure
  (Pure a) >>= f = f a
  (Wrap x) >>= f = Wrap (fmap (>>= f) x)

type Tel = Free TelType

getLine2 :: Tel String
getLine2 = Wrap (GetLine Pure)

putLine2 :: String -> Tel ()
putLine2 s = Wrap (PutLine s (Pure ()))

-- Программа в DSL (не зависит от интерпретатора)
echoUpper :: Tel ()
echoUpper = do
  line <- getLine2
  putLine2 ("Вы ввели: " ++ map toUpper line)
  where toUpper c = if c >= 'a' && c <= 'z'
                    then toEnum (fromEnum c - 32)
                    else c

-- Интерпретатор 1: реальный IO
interpIO :: Tel a -> IO a
interpIO (Pure a) = return a
interpIO (Wrap (GetLine k)) = do
  s <- return "hello world"  -- симуляция ввода
  interpIO (k s)
interpIO (Wrap (PutLine s k)) = do
  putStrLn s
  interpIO k

-- Интерпретатор 2: чистый (для тестов)
interpPure :: [String] -> Tel a -> ([String], a)
interpPure _      (Pure a) = ([], a)
interpPure (i:is) (Wrap (GetLine k)) = interpPure is (k i)
interpPure is     (Wrap (PutLine s k)) =
  let (outputs, result) = interpPure is k
  in (s:outputs, result)
interpPure [] _ = error "no more input"

testFree :: IO ()
testFree = do
  putStrLn "=== IO интерпретатор ==="
  interpIO echoUpper
  putStrLn "\n=== Чистый интерпретатор (для тестов) ==="
  let (outputs, _) = interpPure ["test input"] echoUpper
  mapM_ putStrLn outputs

testFree

## 12. Порядок имеет значение: разные композиции одних и тех же трансформеров

Два стека из **одних и тех же** трансформеров в **разном порядке** дают принципиально разные монады.

| Стек | Тип результата | При сбое/`Nothing` |
|------|----------------|----------------------|
| `MaybeT (State s) a` | `State s (Maybe a)` | состояние **сохраняется** |
| `StateT s Maybe a` | `Maybe (a, s)` | состояние **теряется** |
| `ExceptT e (State s) a` | `State s (Either e a)` | состояние **сохраняется** |
| `StateT s (Either e) a` | `Either e (a, s)` | состояние **теряется** |
| `WriterT w (State s) a` | `State s (a, w)` | состояние внешнее, лог внутри |
| `StateT s (Writer w) a` | `Writer w (a, s)` | лог внешний, состояние внутри |

> **Правило:** при сбое внешнего трансформера пропадает всё, что лежит **внутри** него.
> Читай стек снаружи вовнутрь — это порядок потерь при сбое.

### Визуализация: что теряется при сбое

![Порядок трансформеров](../diagrams/monads/mt_order.svg)

In [ ]:
-- Генерируем SVG-диаграмму сравнения порядков трансформеров
let orderDiagramSvg = unlines
      [ "<svg xmlns='http://www.w3.org/2000/svg' width='640' height='360' font-family='monospace' font-size='13'>"
      , "  <rect width='640' height='360' fill='#1e1e2e' rx='12'/>"
      , "  <text x='320' y='28' fill='#cdd6f4' font-size='14' font-weight='bold' text-anchor='middle'>Порядок трансформеров: что теряется при сбое</text>"
      -- Левая колонка: MaybeT (State s)
      , "  <rect x='15' y='45' width='290' height='140' fill='#313244' rx='8'/>"
      , "  <text x='160' y='65' fill='#a6e3a1' font-size='12' font-weight='bold' text-anchor='middle'>MaybeT (State s) a</text>"
      , "  <text x='160' y='82' fill='#89dceb' font-size='11' text-anchor='middle'>≡  State s (Maybe a)</text>"
      , "  <rect x='35' y='90' width='250' height='22' fill='#45475a' rx='4'/>"
      , "  <text x='160' y='105' fill='#f38ba8' font-size='11' text-anchor='middle'>внешний: MaybeT</text>"
      , "  <rect x='55' y='116' width='210' height='22' fill='#585b70' rx='4'/>"
      , "  <text x='160' y='131' fill='#fab387' font-size='11' text-anchor='middle'>внутренний: State s</text>"
      , "  <text x='160' y='158' fill='#a6e3a1' font-size='11' text-anchor='middle'>✓  состояние живёт при Nothing</text>"
      -- Правая колонка: StateT s Maybe
      , "  <rect x='335' y='45' width='290' height='140' fill='#313244' rx='8'/>"
      , "  <text x='480' y='65' fill='#f38ba8' font-size='12' font-weight='bold' text-anchor='middle'>StateT s Maybe a</text>"
      , "  <text x='480' y='82' fill='#89dceb' font-size='11' text-anchor='middle'>≡  Maybe (a, s)</text>"
      , "  <rect x='355' y='90' width='250' height='22' fill='#45475a' rx='4'/>"
      , "  <text x='480' y='105' fill='#fab387' font-size='11' text-anchor='middle'>внешний: StateT s</text>"
      , "  <rect x='375' y='116' width='210' height='22' fill='#585b70' rx='4'/>"
      , "  <text x='480' y='131' fill='#f38ba8' font-size='11' text-anchor='middle'>внутренний: Maybe</text>"
      , "  <text x='480' y='158' fill='#f38ba8' font-size='11' text-anchor='middle'>✗  состояние гибнет при Nothing</text>"
      -- Вторая пара: ExceptT (State s) vs StateT (Either e)
      , "  <rect x='15' y='200' width='290' height='140' fill='#313244' rx='8'/>"
      , "  <text x='160' y='220' fill='#a6e3a1' font-size='12' font-weight='bold' text-anchor='middle'>ExceptT e (State s) a</text>"
      , "  <text x='160' y='237' fill='#89dceb' font-size='11' text-anchor='middle'>≡  State s (Either e a)</text>"
      , "  <rect x='35' y='245' width='250' height='22' fill='#45475a' rx='4'/>"
      , "  <text x='160' y='260' fill='#f38ba8' font-size='11' text-anchor='middle'>внешний: ExceptT e</text>"
      , "  <rect x='55' y='271' width='210' height='22' fill='#585b70' rx='4'/>"
      , "  <text x='160' y='286' fill='#fab387' font-size='11' text-anchor='middle'>внутренний: State s</text>"
      , "  <text x='160' y='313' fill='#a6e3a1' font-size='11' text-anchor='middle'>✓  состояние живёт при Left e</text>"
      , "  <rect x='335' y='200' width='290' height='140' fill='#313244' rx='8'/>"
      , "  <text x='480' y='220' fill='#f38ba8' font-size='12' font-weight='bold' text-anchor='middle'>StateT s (Either e) a</text>"
      , "  <text x='480' y='237' fill='#89dceb' font-size='11' text-anchor='middle'>≡  Either e (a, s)</text>"
      , "  <rect x='355' y='245' width='250' height='22' fill='#45475a' rx='4'/>"
      , "  <text x='480' y='260' fill='#fab387' font-size='11' text-anchor='middle'>внешний: StateT s</text>"
      , "  <rect x='375' y='271' width='210' height='22' fill='#585b70' rx='4'/>"
      , "  <text x='480' y='286' fill='#f38ba8' font-size='11' text-anchor='middle'>внутренний: Either e</text>"
      , "  <text x='480' y='313' fill='#f38ba8' font-size='11' text-anchor='middle'>✗  состояние гибнет при Left e</text>"
      , "  <text x='320' y='350' fill='#6c7086' font-size='10' text-anchor='middle'>зелёный = состояние выживает · красный = теряется</text>"
      , "</svg>"
      ]
writeFile "/tmp/order_diagram.svg" orderDiagramSvg
putStrLn "order_diagram.svg сохранён"

### Пара 1: `MaybeT (State s)` vs `StateT s Maybe`

**Ключевое различие:** переживает ли накопленное состояние сбой `Nothing`?

```
MaybeT (State s) a  =  s -> (Maybe a, s)   -- состояние всегда возвращается
StateT s Maybe a    =  s -> Maybe (a, s)    -- при Nothing состояния нет
```

In [ ]:
import Control.Monad.Trans.State
import Control.Monad.Trans.Maybe
import Control.Monad.Trans.Class (lift)

-- ── Вариант A: MaybeT (State Int) ───────────────────────────────
-- Тип: State Int (Maybe a)
-- Запуск: runState (runMaybeT m) startState :: (Maybe a, Int)
-- Особенность: состояние продолжает изменяться даже при Nothing
type MaybeState s a = MaybeT (State s) a

-- Успешное вычисление
stepA_ok :: MaybeState Int String
stepA_ok = do
  lift $ modify (+1)        -- 0 → 1
  lift $ modify (+4)        -- 1 → 5
  s <- lift get
  return ("значение: " ++ show s)

-- Вычисление с частичным изменением состояния до сбоя
stepA_fail :: MaybeState Int String
stepA_fail = do
  lift $ modify (+1)        -- 0 → 1 (это изменение СОХРАНИТСЯ)
  lift $ modify (+1)        -- 1 → 2 (и это тоже)
  MaybeT (return Nothing)   -- сбой — но состояние = 2 остаётся!

-- ── Вариант B: StateT Int Maybe ──────────────────────────────────
-- Тип: Maybe (a, Int)
-- Запуск: runStateT m startState :: Maybe (a, Int)
-- Особенность: при Nothing весь результат = Nothing, нет состояния
type StateMaybe s a = StateT s Maybe a

stepB_ok :: StateMaybe Int String
stepB_ok = do
  modify (+1)
  modify (+4)
  s <- get
  return ("значение: " ++ show s)

stepB_fail :: StateMaybe Int String
stepB_fail = do
  modify (+1)   -- попытка изменить: 0 → 1
  modify (+1)   -- попытка: 1 → 2
  lift Nothing  -- сбой → весь результат Nothing, состояние исчезло

-- ── Демонстрация ─────────────────────────────────────────────────
demoMaybeVsState :: IO ()
demoMaybeVsState = do
  putStrLn "=== MaybeT (State Int) : состояние СОХРАНЯЕТСЯ ==="
  let (r1, s1) = runState (runMaybeT stepA_ok)   0
  putStrLn $ "  ok:   result=" ++ show r1 ++ "  state=" ++ show s1
  let (r2, s2) = runState (runMaybeT stepA_fail) 0
  putStrLn $ "  fail: result=" ++ show r2 ++ "  state=" ++ show s2
  putStrLn   "        ^ состояние = 2, хотя результат Nothing!"

  putStrLn ""
  putStrLn "=== StateT Int Maybe : состояние ТЕРЯЕТСЯ ==="
  putStrLn $ "  ok:   " ++ show (runStateT stepB_ok   0)
  putStrLn $ "  fail: " ++ show (runStateT stepB_fail 0)
  putStrLn   "        ^ Nothing — никакого состояния"

demoMaybeVsState

### Пара 2: `ExceptT e (State s)` vs `StateT s (Either e)`

Та же история, что и с `Maybe`, но ошибка несёт информацию `e`.

Практический пример: **транзакция с аудитом**.
Если нужно знать, *до какого шага* дошло вычисление — берём `ExceptT` снаружи.
Если нужен явный откат состояния — берём `StateT` снаружи.

In [ ]:
import Control.Monad.Trans.Except

-- ── Вариант A: ExceptT String (State Int) ───────────────────────
-- Тип: State Int (Either String a)
-- Состояние выживает при throwE — можно сделать аудит
type ExceptState s e a = ExceptT e (State s) a

bankTransfer :: ExceptState Int String Int
bankTransfer = do
  lift $ modify (+100)         -- поступление +100
  lift $ modify (+50)          -- поступление +50
  bal <- lift get              -- баланс = 150
  if bal > 120
    then throwE ("Превышен лимит! Баланс=" ++ show bal)
    else return bal

runES :: Int -> ExceptState Int String a -> (Either String a, Int)
runES s0 m = runState (runExceptT m) s0

-- ── Вариант B: StateT Int (Either String) ───────────────────────
-- Тип: Either String (a, Int)
-- При Left — состояния нет вообще
type StateExcept s e a = StateT s (Either e) a

bankTransferB :: StateExcept Int String Int
bankTransferB = do
  modify (+100)
  modify (+50)
  bal <- get
  if bal > 120
    then StateT (\_ -> Left ("Превышен лимит! Баланс=" ++ show bal))
    else return bal

runSE :: Int -> StateExcept Int String a -> Either String (a, Int)
runSE s0 m = runStateT m s0

-- ── Демонстрация ─────────────────────────────────────────────────
demoExceptVsState :: IO ()
demoExceptVsState = do
  putStrLn "=== ExceptT String (State Int): состояние ДОСТУПНО при ошибке ==="
  let (r1, s1) = runES 0 bankTransfer
  putStrLn $ "  результат: " ++ show r1
  putStrLn $ "  состояние (баланс): " ++ show s1
  putStrLn   "  (видно что транзакции прошли, просто лимит превышен)"

  putStrLn ""
  putStrLn "=== StateT Int (Either String): состояние ПРОПАЛО ==="
  let r2 = runSE 0 bankTransferB
  putStrLn $ "  результат: " ++ show r2
  putStrLn   "  (нет информации о состоянии — только ошибка)"

  putStrLn ""
  putStrLn "Вывод: ExceptT снаружи = состояние для аудита,"
  putStrLn "       StateT снаружи  = автоматический откат при ошибке."

demoExceptVsState

### Пара 3: `WriterT w (State s)` vs `StateT s (Writer w)`

Оба комбинируют изменяемое состояние и накопление лога.
Разница — только в **форме вложения** возвращаемого типа.
Семантически они эквивалентны (ни один не «пропадает» при сбое — здесь сбоев нет).

```
WriterT [String] (State Int) a  ↔  runState (runWriterT m) s0 :: ((a,[String]), Int)
StateT Int (Writer [String]) a  ↔  runWriter (runStateT m s0) :: ((a, Int), [String])
```

In [ ]:
import Control.Monad.Trans.Writer

-- ── Вариант A: WriterT [String] (State Int) ─────────────────────
-- "Снаружи — лог, внутри — счётчик"
-- runState (runWriterT m) s0  :: ((a, [String]), Int)
type LogState a = WriterT [String] (State Int) a

processA :: [Int] -> LogState [Int]
processA = mapM step
  where
    step x = do
      lift $ modify (+1)
      n <- lift get
      tell ["[шаг " ++ show n ++ "] обработан " ++ show x]
      return (x * 2)

runLogState :: Int -> LogState a -> ((a, [String]), Int)
runLogState s0 m = runState (runWriterT m) s0

-- ── Вариант B: StateT Int (Writer [String]) ─────────────────────
-- "Снаружи — счётчик, внутри — лог"
-- runWriter (runStateT m s0) :: ((a, Int), [String])
type StateLog a = StateT Int (Writer [String]) a

processB :: [Int] -> StateLog [Int]
processB = mapM step
  where
    step x = do
      modify (+1)
      n <- get
      lift $ tell ["[шаг " ++ show n ++ "] обработан " ++ show x]
      return (x * 2)

runStateLog :: Int -> StateLog a -> ((a, Int), [String])
runStateLog s0 m = runWriter (runStateT m s0)

-- ── Демонстрация ─────────────────────────────────────────────────
demoWriterState :: IO ()
demoWriterState = do
  let items = [10, 20, 30]

  putStrLn "=== WriterT [String] (State Int) ==="
  let ((resA, logA), stA) = runLogState 0 (processA items)
  putStrLn $ "  результаты: " ++ show resA ++ "  финальный счётчик: " ++ show stA
  mapM_ (\l -> putStrLn $ "  лог: " ++ l) logA

  putStrLn ""
  putStrLn "=== StateT Int (Writer [String]) ==="
  let ((resB, stB), logB) = runStateLog 0 (processB items)
  putStrLn $ "  результаты: " ++ show resB ++ "  финальный счётчик: " ++ show stB
  mapM_ (\l -> putStrLn $ "  лог: " ++ l) logB

  putStrLn ""
  putStrLn "Результаты идентичны — разница только в скобках типа."

demoWriterState

### Пара 4: `ReaderT r Maybe` vs `MaybeT (Reader r)`

`Reader` — это чистая монада (`r -> a`), без побочных эффектов.
Поэтому при перестановке с `Maybe` тип **не меняется**:

```
ReaderT r Maybe a  =  r -> Maybe a
MaybeT (Reader r) a  =  Reader r (Maybe a)  =  r -> Maybe a   -- то же самое!
```

Это **особый случай**: когда внешняя монада чистая, порядок семантически не важен.
Разница — только в том, как писать `lift` при доступе к окружению.

In [ ]:
import qualified Data.Map.Strict as Map
import Control.Monad.Trans.Reader (runReader)

type Env = Map.Map String Int

-- ── Вариант A: ReaderT Env Maybe ────────────────────────────────
-- ask доступен напрямую, lift нужен для Maybe-операций
type ReaderMaybe a = ReaderT Env Maybe a

lookupA :: String -> ReaderMaybe Int
lookupA name = do
  env <- ask
  lift $ Map.lookup name env   -- lift :: Maybe a -> ReaderMaybe a

calcA :: ReaderMaybe Int
calcA = do
  x <- lookupA "x"
  y <- lookupA "y"
  return (x + y)

-- ── Вариант B: MaybeT (Reader Env) ──────────────────────────────
-- ask требует lift, MaybeT-операции через MaybeT . return
type MaybeReader a = MaybeT (Reader Env) a

lookupB :: String -> MaybeReader Int
lookupB name = do
  env <- lift ask              -- lift :: Reader Env a -> MaybeReader a
  MaybeT $ return $ Map.lookup name env

calcB :: MaybeReader Int
calcB = do
  x <- lookupB "x"
  y <- lookupB "y"
  return (x + y)

-- ── Демонстрация ─────────────────────────────────────────────────
demoReaderMaybe :: IO ()
demoReaderMaybe = do
  let envFull    = Map.fromList [("x", 10), ("y", 20)]
  let envMissing = Map.fromList [("x", 10)]          -- y отсутствует

  putStrLn "=== ReaderT Env Maybe ==="
  putStrLn $ "  полный env:   " ++ show (runReaderT calcA envFull)
  putStrLn $ "  неполный env: " ++ show (runReaderT calcA envMissing)

  putStrLn ""
  putStrLn "=== MaybeT (Reader Env) ==="
  putStrLn $ "  полный env:   " ++ show (runReader (runMaybeT calcB) envFull)
  putStrLn $ "  неполный env: " ++ show (runReader (runMaybeT calcB) envMissing)

  putStrLn ""
  putStrLn "Оба варианта: r -> Maybe a  — семантически тождественны."
  putStrLn "Выбор зависит от стиля: где удобнее писать lift."

demoReaderMaybe

### Пара 5: IO — всегда в основании стека

`IO` не является трансформером, поэтому она **всегда самая внутренняя** монада в стеке.
Не существует `IOT m a` — нельзя «засунуть IO внутрь» другой монады.

Типичные production-стеки (снаружи вовнутрь):

```
ReaderT Config (ExceptT AppError IO) a  — конфиг + ошибки + IO
StateT s (ExceptT e IO) a               — состояние + ошибки + IO  
WriterT [Log] (ReaderT Env IO) a        — лог + конфиг + IO
```

In [ ]:
-- Production-стек: ReaderT Config (ExceptT AppError IO) a
-- Это самый распространённый паттерн в Haskell-приложениях
import Control.Monad (when)


data AppError2 = NotFound2 String | DBError2 String | Unauthorized2
  deriving Show

data Config2 = Config2 { dbUrl :: String, debug2 :: Bool }

type App2 a = ReaderT Config2 (ExceptT AppError2 IO) a

-- Запускаем стек: снимаем ReaderT, потом ExceptT
runApp2 :: Config2 -> App2 a -> IO (Either AppError2 a)
runApp2 cfg m = runExceptT (runReaderT m cfg)

-- Вспомогательные функции
throwApp2 :: AppError2 -> App2 a
throwApp2 = lift . throwE

logDebug2 :: String -> App2 ()
logDebug2 msg = do
  cfg <- ask
  when (debug2 cfg) $ liftIO $ putStrLn $ "[DEBUG] " ++ msg

-- Симуляция запроса к БД
dbQuery :: String -> App2 [String]
dbQuery q = do
  cfg <- ask
  logDebug2 $ "Query: " ++ q
  if null (dbUrl cfg)
    then throwApp2 (DBError2 "нет подключения")
    else do
      liftIO $ putStrLn $ "  → " ++ dbUrl cfg ++ " :: " ++ q
      return ["row1", "row2"]

-- Бизнес-логика
getUser :: Int -> App2 String
getUser uid
  | uid <= 0  = throwApp2 (NotFound2 $ "uid=" ++ show uid)
  | otherwise = do
      rows <- dbQuery ("SELECT name FROM users WHERE id=" ++ show uid)
      return (head rows)

-- ── Тест ─────────────────────────────────────────────────────────
testApp2 :: IO ()
testApp2 = do
  let cfg = Config2 "postgres://db:5432/app" True

  putStrLn "=== Успешный запрос ==="
  r1 <- runApp2 cfg (getUser 42)
  putStrLn $ "Результат: " ++ show r1

  putStrLn "\n=== NotFound ==="
  r2 <- runApp2 cfg (getUser (-1))
  putStrLn $ "Результат: " ++ show r2

  putStrLn "\n=== DBError (нет URL) ==="
  r3 <- runApp2 (cfg { dbUrl = "" }) (getUser 42)
  putStrLn $ "Результат: " ++ show r3

testApp2

### Резюме: как выбирать порядок трансформеров

#### Правило «внешнее пропадает»

При сбое **внешнего** трансформера всё содержимое **внутренних** слоёв исчезает.

![Внешний/внутренний трансформер](../diagrams/monads/mt_layers.svg)

#### Таблица выбора

| Хочу... | Стек |
|---------|------|
| Состояние пережило сбой | `ExceptT e (State s)` / `MaybeT (State s)` |
| Автоматический откат состояния | `StateT s (Either e)` / `StateT s Maybe` |
| Лог пережил сбой (MaybeT + Writer) | `MaybeT (WriterT w m)` |
| Production: конфиг + ошибки + IO | `ReaderT cfg (ExceptT e IO)` |
| Аудит: видеть состояние после ошибки | `ExceptT e (StateT s IO)` |

#### Мнемоника

Читай стек **снаружи вовнутрь** — это то, что **пропадает первым** при сбое.
Читай стек **изнутри наружу** — это то, что **выживает дольше всего**.

← [ComonadTransformers.ipynb](ComonadTransformers.ipynb) — Трансформеры комонад


---
**← Предыдущий:** [Monads](Monads.ipynb)  |  **Следующий →** [Comonads](Comonads.ipynb)